# Setup

In [26]:
%pip install xxhash pandas matplotlib numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import uuid
from random import randint, sample, shuffle
from typing import Dict, List, Optional
from abc import ABC, abstractmethod

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anchorhash import AnchorHasher
from mementohash import MementoHasher
from dxhash_lmf import DxHasher

In [3]:
class Hascher(ABC):
    @abstractmethod
    def getShard(self, record_id: str) -> int:
        pass

    @abstractmethod
    def addShard(self, shard_id: int):
        pass

    @abstractmethod
    def dropShard(self, shard_id: int):
        pass

In [4]:
class AnchorHasherAdapter(Hascher):
    def __init__(
        self, working_set: List[str], capacity: int, seed: Optional[int] = None
    ) -> None:
        self._impl = AnchorHasher(working_set, capacity=capacity, seed=seed)
        self.name = self._impl.name

    def getShard(self, record_id: str) -> int:
        return self._impl.getShard(record_id)

    def addShard(self, shard_id: int) -> None:
        self._impl.addShard(shard_id)

    def dropShard(self, shard_id: int) -> None:
        self._impl.dropShard(shard_id)

In [5]:
class MementoHasherAdapter(Hascher):
    def __init__(
        self, working_set: int, capacity: int, seed: Optional[int] = None
    ) -> None:
        self._impl = MementoHasher(working_set, capacity=capacity, seed=seed)
        self.name = self._impl.name

    def getShard(self, record_id: str) -> int:
        return self._impl.getShard(record_id)

    def addShard(self, shard_id: int) -> None:
        self._impl.addShard(shard_id)

    def dropShard(self, shard_id: int) -> None:
        self._impl.dropShard(shard_id)

In [6]:
class DxHasherAdapter(Hascher):
    def __init__(
        self, working_set: int, capacity: int, seed: Optional[int] = None
    ) -> None:
        self._impl = DxHasher(working_set, capacity=capacity, seed=seed) 
        self.name = "DXHash"

    def getShard(self, record_id: str) -> int:
        return self._impl.getShard(record_id)

    def addShard(self, shard_id: int) -> None:
        self._impl.addShard(shard_id)

    def dropShard(self, shard_id: int) -> None:
        self._impl.dropShard(shard_id)

In [30]:
def generate_removal_different_places(original_order, distance):
    n = len(original_order)
    scramble_count = int(n * distance)
    map = {}

    for i in range(n):
        if i+scramble_count > n:
            break

        # Split the array into three parts
        before = original_order[:i]
        middle = original_order[i:i+scramble_count]
        after = original_order[i+scramble_count:]

        # Shuffle the middle part
        shuffle(middle)

        # Combine the parts back together
        new_order = before + middle + after

        # Save the new order in the map with the starting index as key
        map[i] = new_order
    
    return map 

def generate_records(count: int = 100000) -> List[uuid.UUID]:
    """Generate a list of random records for testing."""
    return [uuid.uuid1() for i in range(count)]

def run_single_distance_experiment(
    h_default: Hascher, h_variant: Hascher, records: List, distance: float
) -> float:
    """Run experiment for single distance value and return failure rate."""
    mismatches = 0
    for r in records:
        if h_default.getShard(str(r)) != h_variant.getShard(str(r)):
            mismatches += 1
    failure_rate = mismatches / len(records)
    print(f"distance={distance:.2f} mismatches={mismatches} rate={failure_rate:.4f}")
    return failure_rate

# Experiment

In [ ]:
def run_experiment(RECORDS: List, map: Dict[int, List[int]], total_shards: int, dropped_shards: List[int]):
    indexes = sorted(map.keys())
    results = {}

    for d in indexes:
        diff_order = map[d]
        seed = randint(0, 2**32 - 1)

        # Create hasher instances
        h_default = MementoHasherAdapter(
            working_set=total_shards, capacity=total_shards, seed=seed
        )
        h_variant = MementoHasherAdapter(
            working_set=total_shards, capacity=total_shards, seed=seed
        )
        # Apply removals
        for s in reversed(dropped_shards):
            h_default.dropShard(s)
        for s in reversed(diff_order):
            h_variant.dropShard(s)

        # Calculate and store failure rate
        failure_rate = run_single_distance_experiment(
            h_default, h_variant, RECORDS, d
        )
        
        results[d] = {
            "index": d,
            "failure_rates": failure_rate,
            "dropped_shards": dropped_shards,
        }
    return results

In [ ]:
def run_single_size(RECORDS: List, total_shards: int, drop_shards_cnt: int):
    results = {}

    for i in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        dropped_shards = sample(range(total_shards), drop_shards_cnt)

        map = generate_removal_different_places(dropped_shards, i)
        result = run_experiment(RECORDS, map, total_shards, dropped_shards)
        results[i] = result
    
    return results

In [33]:
RECORDS = generate_records(10000)

total_shards = 1024
results = {}

for i in [0.25, 0.5, 0.75]:
    drop_shards_cnt = int(total_shards * i)
    results[i] = run_single_size(RECORDS, total_shards, drop_shards_cnt)

# Plot the results
for n, res in results.items():
    rows = []

    for i, fig in res.items():  # your loop
        index_values = [result["index"] for result in fig.values()]
        failure_rates = [result["failure_rates"] for result in fig.values()]
        
        for idx, fail_rate in zip(index_values, failure_rates):
            rows.append({
                'index': idx,
                'failurerate': fail_rate,
                'distance': i
            })

    df = pd.DataFrame(rows)
    df.to_csv(f"anchorhash_removalrate_{n}.csv", index=False)

# Plot the results
for n, res in results.items():
    for i, fig in res.items():
        plt.figure(figsize=(8, 4))
        # Extract data for plotting
        distances = [result["index"] for result in fig.values()]
        failure_rates = [result["failure_rates"] for result in fig.values()]

        plt.plot(
            distances,
            failure_rates,
            marker="o",
            label=f"Dropped {n*100:.0f}% of shards",
        )

    plt.xlabel("Index")
    plt.ylabel("Failure Rate")
    plt.title("AnchorHash: Failure Rate vs Index")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend()
    plt.show()

distance=0.00 mismatches=339 rate=0.0339
distance=1.00 mismatches=292 rate=0.0292
distance=2.00 mismatches=279 rate=0.0279
distance=3.00 mismatches=266 rate=0.0266
distance=4.00 mismatches=319 rate=0.0319
distance=5.00 mismatches=321 rate=0.0321
distance=6.00 mismatches=311 rate=0.0311
distance=7.00 mismatches=330 rate=0.0330
distance=8.00 mismatches=281 rate=0.0281
distance=9.00 mismatches=276 rate=0.0276
distance=10.00 mismatches=287 rate=0.0287
distance=11.00 mismatches=284 rate=0.0284
distance=12.00 mismatches=290 rate=0.0290
distance=13.00 mismatches=323 rate=0.0323
distance=14.00 mismatches=311 rate=0.0311
distance=15.00 mismatches=320 rate=0.0320
distance=16.00 mismatches=335 rate=0.0335
distance=17.00 mismatches=308 rate=0.0308
distance=18.00 mismatches=299 rate=0.0299
distance=19.00 mismatches=298 rate=0.0298
distance=20.00 mismatches=339 rate=0.0339
distance=21.00 mismatches=316 rate=0.0316
distance=22.00 mismatches=296 rate=0.0296
distance=23.00 mismatches=314 rate=0.0314
di

KeyboardInterrupt: 